# OJAS — Real Ocean Acoustic Dataset

We start from real ocean observations:

- Depth
- Temperature
- Salinity
- Turbidity
- Measured/Mackenzie sound speed

The missing acoustic parameters will be generated using a physics-based
candidate-waveform model.

Important:
The generated acoustic parameters are model-derived, not direct measurements.

In [63]:
import pandas as pd
import numpy as np

# Load your uploaded CSV
df = pd.read_csv("/content/ocean_data.csv")

# Remove the accidental CSV index column
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

display(df.head())

Rows: 1839
Columns:
['depth_m', 'temperature_C', 'salinity_PSU', 'turbidity_NTU', 'sound_speed_mps_mackenzie', 'sound_speed_data_mps', 'sound_speed_delta_mps', 'profile_depth_max_m']


,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m
0,1.0,14.41920,24.17840,0.5541,1491.957290,1473.875,18.082290,21.0
1,2.0,14.51465,34.82145,0.5238,1504.968400,1473.875,31.093400,21.0
2,5.0,14.26420,35.93365,0.7444,1505.540125,1473.875,31.665125,21.0
3,10.0,14.11650,35.96055,0.8524,1505.178267,1473.875,31.303267,21.0
4,20.0,14.07870,35.98490,1.2605,1505.248443,1473.875,31.373443,21.0


## Acoustic Scenario Configuration

The ocean dataset alone cannot determine the transmitted waveform.

We therefore define the mission/hardware assumptions below.

Target range:

R = target distance from transmitter to target

Target strength:

TS = target acoustic reflectivity

Ambient noise:

NL_PSD = noise spectral density

Source level:

SL = transmitted acoustic source level

These are scenario parameters and should eventually be replaced by
your actual AUV/transducer specifications.

In [64]:
# ==============================
# OJAS ACOUSTIC CONFIGURATION
# ==============================

TARGET_RANGE_M = 100.0

# Target strength assumption
TARGET_STRENGTH_DB = -10.0

# Ambient noise PSD baseline.
# This is a PLACEHOLDER until you insert NOAA measured PSD.
NOISE_PSD_BASE_DB = 60.0

# Reference source level of the acoustic transmitter.
# Replace with your transducer's measured/calibrated value.
SOURCE_LEVEL_DB = 180.0

# Required detection margin
MIN_DETECTION_MARGIN_DB = 10.0

# Hardware limits
MIN_FREQUENCY_HZ = 50_000
MAX_FREQUENCY_HZ = 600_000

MIN_BANDWIDTH_HZ = 10_000
MAX_BANDWIDTH_HZ = 20_000

MAX_ENERGY_J = 1.0

# Processing/sample-rate assumption
SAMPLE_RATE_HZ = 2_00_000

print("Acoustic configuration loaded.")

Acoustic configuration loaded.


In [65]:
# Approximate hydrostatic pressure
df["pressure_dbar"] = df["depth_m"] / 10.1

print(df[["depth_m", "pressure_dbar"]].head())

   depth_m  pressure_dbar
0      1.0       0.099010
1      2.0       0.198020
2      5.0       0.495050
3     10.0       0.990099
4     20.0       1.980198


Acoustic absorption

For the first version, use the Thorp absorption equation.

For frequency \(f\) in kHz:

$$ \alpha(f)= \frac{0.11f^2}{1+f^2} + \frac{44f^2}{4100+f^2} + 2.75\times10^{-4}f^2 + 0.003 $$

where:

$$ \alpha = \text{absorption in dB/km} $$

This gives us a frequency-dependent acoustic attenuation.

Important: this is a simplified model. It does not use your temperature, salinity or pH. A later version can replace it with a more detailed absorption model once pH is available.

In [66]:
REFERENCE_FREQUENCY_HZ = 100_000

f_khz = REFERENCE_FREQUENCY_HZ / 1000.0

alpha = (
    0.11 * f_khz**2 / (1 + f_khz**2)
    + 44 * f_khz**2 / (4100 + f_khz**2)
    + 2.75e-4 * f_khz**2
    + 0.003
)

df["absorption_dB_per_km"] = alpha

print("Reference absorption:", alpha, "dB/km")

Reference absorption: 34.068662759965136 dB/km


Transmission loss

Use a simple spreading + absorption model:

$$ TL(R,f)=20\log_{10}(R)+\alpha(f)\frac{R}{1000} $$

where:

\(R\) = range in metres
\(\alpha\) = absorption in dB/km

For an active echo, propagation occurs approximately twice, so:

TL
active
	​

=2TL

In [67]:
R = TARGET_RANGE_M

TL = (
    20 * np.log10(np.maximum(R, 1.0))
    + df["absorption_dB_per_km"] * (R / 1000.0)
)

df["transmission_loss_dB"] = TL

display(df[["depth_m", "transmission_loss_dB"]].head())

,depth_m,transmission_loss_dB
0,1.0,43.406866
1,2.0,43.406866
2,5.0,43.406866
3,10.0,43.406866
4,20.0,43.406866


Turbidity/scattering proxy

Turbidity should not be inserted directly into the sound-speed equation.

Instead we create a separate empirical scattering/reverberation proxy:

$$ \eta_s = k_t \cdot Turbidity \cdot \left(\frac{f}{f_{ref}}\right)^\gamma $$

This is a proxy, not a universal physical acoustic attenuation law.

It allows your real turbidity measurements to influence waveform selection until you have actual acoustic scattering measurements for calibration.

In [68]:
# Reference turbidity
TURBIDITY_REF_NTU = 1.0

# Reference frequency
SCATTERING_REF_HZ = 100_000

# Empirical scaling parameters
SCATTERING_K = 0.05
SCATTERING_GAMMA = 1.5

eta_s = (
    SCATTERING_K
    * df["turbidity_NTU"]
    / TURBIDITY_REF_NTU
    * (REFERENCE_FREQUENCY_HZ / SCATTERING_REF_HZ)
    ** SCATTERING_GAMMA
)

df["scattering_index"] = eta_s

Candidate waveform library

We now create the waveform design space.

The ocean does not directly tell us:

$$ f_c,\ B,\ T_p,\ A $$

Instead, we test candidate combinations.


In [69]:
# ============================================================
# CELL 15 — Expanded Acoustic Waveform Candidate Library
# ============================================================

candidates = pd.DataFrame({
    "candidate_id": [
        "C01","C02","C03","C04","C05","C06","C07","C08",
        "C09","C10","C11","C12","C13","C14","C15","C16",
        "C17","C18","C19","C20","C21","C22","C23","C24"
    ],

    "waveform_type": [
        # LFM
        "LFM","LFM","LFM","LFM","LFM","LFM","LFM","LFM",

        # GSFM
        "GSFM","GSFM","GSFM","GSFM","GSFM","GSFM","GSFM","GSFM",

        # Phase-coded
        "Phase-coded","Phase-coded","Phase-coded","Phase-coded",
        "Phase-coded","Phase-coded","Phase-coded","Phase-coded"
    ],

    "center_frequency_hz": [
        # LFM
        60_000, 100_000, 150_000, 200_000,
        300_000, 400_000, 500_000, 590_000,

        # GSFM
        60_000, 100_000, 150_000, 200_000,
        300_000, 400_000, 500_000, 590_000,

        # Phase-coded
        60_000, 100_000, 150_000, 200_000,
        300_000, 400_000, 500_000, 590_000
    ],

    "bandwidth_hz": [
        # LFM
        10_000, 12_000, 14_000, 16_000,
        18_000, 20_000, 20_000, 20_000,

        # GSFM
        20_000, 18_000, 16_000, 14_000,
        12_000, 10_000, 18_000, 20_000,

        # Phase-coded
        10_000, 12_000, 14_000, 16_000,
        18_000, 20_000, 18_000, 20_000
    ],

    "pulse_duration_s": [
        # LFM
        0.005, 0.008, 0.010, 0.012,
        0.015, 0.018, 0.020, 0.020,

        # GSFM
        0.005, 0.008, 0.010, 0.012,
        0.015, 0.018, 0.020, 0.020,

        # Phase-coded
        0.005, 0.008, 0.010, 0.012,
        0.015, 0.018, 0.020, 0.020
    ],

    "amplitude": [
        # LFM
        0.80, 0.80, 0.80, 0.80,
        0.80, 0.80, 0.80, 0.80,

        # GSFM
        0.80, 0.80, 0.80, 0.80,
        0.80, 0.80, 0.80, 0.80,

        # Phase-coded
        0.75, 0.75, 0.75, 0.75,
        0.75, 0.75, 0.75, 0.75
    ],

    "phase_code": [
        # LFM
        "none","none","none","none",
        "none","none","none","none",

        # GSFM
        "none","none","none","none",
        "none","none","none","none",

        # Phase-coded
        "Barker-7","Barker-7","Barker-7","Barker-13",
        "Barker-13","Barker-13","Barker-13","Barker-13"
    ]
})

# ------------------------------------------------------------
# Check candidate library
# ------------------------------------------------------------

print("Number of candidates:", len(candidates))
display(candidates)

Number of candidates: 24


,candidate_id,waveform_type,center_frequency_hz,bandwidth_hz,pulse_duration_s,amplitude,phase_code
0,C01,LFM,60000,10000,0.005,0.80,none
1,C02,LFM,100000,12000,0.008,0.80,none
2,C03,LFM,150000,14000,0.010,0.80,none
3,C04,LFM,200000,16000,0.012,0.80,none
4,C05,LFM,300000,18000,0.015,0.80,none
5,C06,LFM,400000,20000,0.018,0.80,none
6,C07,LFM,500000,20000,0.020,0.80,none
7,C08,LFM,590000,20000,0.020,0.80,none
8,C09,GSFM,60000,20000,0.005,0.80,none
9,C10,GSFM,100000,18000,0.008,0.80,none


Candidate absorption

For each candidate:

$$ \alpha_k=\alpha(f_{c,k}) $$

Therefore the same ocean environment can have different attenuation for different candidate frequencies.

This is the critical connection:

Ocean condition+f
c
→frequency dependent attenuation
	​


In [70]:
# ============================================================
# CELL 16 — Frequency-dependent Absorption for Each Candidate
# ============================================================

def thorp_absorption(f_hz):
    """
    Thorp absorption model.

    Input:
        f_hz : frequency in Hz

    Output:
        absorption in dB/km
    """

    f_khz = f_hz / 1000.0

    alpha = (
        0.11 * f_khz**2 / (1 + f_khz**2)
        + 44 * f_khz**2 / (4100 + f_khz**2)
        + 2.75e-4 * f_khz**2
        + 0.003
    )

    return alpha


# Calculate absorption for every candidate
candidates["absorption_dB_per_km"] = (
    candidates["center_frequency_hz"]
    .apply(thorp_absorption)
)

display(
    candidates[
        [
            "candidate_id",
            "waveform_type",
            "center_frequency_hz",
            "bandwidth_hz",
            "absorption_dB_per_km"
        ]
    ]
)

,candidate_id,waveform_type,center_frequency_hz,bandwidth_hz,absorption_dB_per_km
0,C01,LFM,60000,10000,21.674398
1,C02,LFM,100000,12000,34.068663
2,C03,LFM,150000,14000,43.518540
3,C04,LFM,200000,16000,51.022294
4,C05,LFM,300000,18000,66.945889
5,C06,LFM,400000,20000,87.013670
6,C07,LFM,500000,20000,112.153043
7,C08,LFM,590000,20000,139.328291
8,C09,GSFM,60000,20000,21.674398
9,C10,GSFM,100000,18000,34.068663


In [71]:
# ============================================================
# CELL 17 — Create Ocean × Candidate Matrices
# ============================================================

C = df["sound_speed_mps_mackenzie"].values[:, None]

Turb = df["turbidity_NTU"].values[:, None]

F = candidates["center_frequency_hz"].values[None, :]

B = candidates["bandwidth_hz"].values[None, :]

Tp = candidates["pulse_duration_s"].values[None, :]

Amp = candidates["amplitude"].values[None, :]

Alpha = candidates["absorption_dB_per_km"].values[None, :]

# Range resolution for every ocean row × candidate
range_resolution = C / (2 * B)

print("Ocean rows:", len(df))
print("Candidates:", len(candidates))
print("Evaluation matrix shape:", range_resolution.shape)

Ocean rows: 1839
Candidates: 24
Evaluation matrix shape: (1839, 24)


Candidate transmission loss

For each candidate:

$$ TL_k= 20\log_{10}(R) + \alpha_k\frac{R}{1000} $$

Then include your turbidity/scattering proxy:

TL
total,k
	​

=TL
k
	​

+η
s,k
	​


In [72]:
R = TARGET_RANGE_M

candidates["transmission_loss_dB"] = (
    20 * np.log10(R)
    + candidates["absorption_dB_per_km"] * (R / 1000.0)
)

# Frequency-dependent scattering penalty
candidates["scattering_factor"] = (
    (candidates["center_frequency_hz"] / SCATTERING_REF_HZ)
    ** SCATTERING_GAMMA
)

# Store the candidate-level scattering coefficient.
# Actual row-specific scattering is calculated during evaluation.
display(candidates)

,candidate_id,waveform_type,center_frequency_hz,bandwidth_hz,pulse_duration_s,amplitude,phase_code,absorption_dB_per_km,transmission_loss_dB,scattering_factor
0,C01,LFM,60000,10000,0.005,0.80,none,21.674398,42.167440,0.464758
1,C02,LFM,100000,12000,0.008,0.80,none,34.068663,43.406866,1.000000
2,C03,LFM,150000,14000,0.010,0.80,none,43.518540,44.351854,1.837117
3,C04,LFM,200000,16000,0.012,0.80,none,51.022294,45.102229,2.828427
4,C05,LFM,300000,18000,0.015,0.80,none,66.945889,46.694589,5.196152
5,C06,LFM,400000,20000,0.018,0.80,none,87.013670,48.701367,8.000000
6,C07,LFM,500000,20000,0.020,0.80,none,112.153043,51.215304,11.180340
7,C08,LFM,590000,20000,0.020,0.80,none,139.328291,53.932829,14.331050
8,C09,GSFM,60000,20000,0.005,0.80,none,21.674398,42.167440,0.464758
9,C10,GSFM,100000,18000,0.008,0.80,none,34.068663,43.406866,1.000000


Range resolution

Your measured sound speed now becomes directly useful.

$$ \boxed{ \Delta R=\frac{c}{2B} } $$

where:

\(c\) = measured/derived sound speed
\(B\) = candidate bandwidth

Therefore every ocean row gets a different range resolution for every candidate.

In [73]:
# Matrix dimensions:
# rows    = ocean observations
# columns = waveform candidates

C = df["sound_speed_mps_mackenzie"].values[:, None]
Turb = df["turbidity_NTU"].values[:, None]

F = candidates["center_frequency_hz"].values[None, :]
B = candidates["bandwidth_hz"].values[None, :]
Tp = candidates["pulse_duration_s"].values[None, :]
Amp = candidates["amplitude"].values[None, :]
Alpha = candidates["absorption_dB_per_km"].values[None, :]

# Range resolution
range_resolution = C / (2 * B)

print("Candidate evaluation matrix:", range_resolution.shape)

Candidate evaluation matrix: (1839, 24)


Received signal level

For active sonar:

$$ RL_k = SL_k - 2TL_k + TS $$

where:

$$ SL_k=SL_{ref}+20\log_{10}(A_k) $$

So amplitude affects source level.

In [74]:
# Source level for each candidate
SL = SOURCE_LEVEL_DB + 20 * np.log10(Amp)

# Basic propagation loss
TL = (
    20 * np.log10(R)
    + Alpha * (R / 1000.0)
)

# Turbidity-dependent scattering penalty
SCATTERING_LOSS = (
    SCATTERING_K
    * Turb
    * (F / SCATTERING_REF_HZ) ** SCATTERING_GAMMA
)

TOTAL_TL = TL + SCATTERING_LOSS

# Active sonar received echo level
RL = SL - 2 * TOTAL_TL + TARGET_STRENGTH_DB

Noise and SNR

Noise PSD is converted into noise over the waveform bandwidth:

$$ NL_B=NL_{PSD}+10\log_{10}(B) $$

Then:

$$ SNR=RL-NL_B $$

We additionally use pulse compression/processing gain:

$$ G_p\approx10\log_{10}(BT_p) $$

Therefore:

SNR
processed
= RL − NLB + 10 log(BTp)

In [75]:
# Row-specific noise proxy from turbidity.
# Replace this later with NOAA measured noise PSD.
noise_psd = (
    NOISE_PSD_BASE_DB
    + 10 * np.log10(1 + Turb)
)

# Integrated noise in candidate bandwidth
noise_band = noise_psd + 10 * np.log10(B)

# Processing gain
processing_gain = 10 * np.log10(B * Tp)

# Processed SNR
SNR = RL - noise_band + processing_gain

# Detection margin
detection_margin = SNR - MIN_DETECTION_MARGIN_DB

print("SNR matrix:", SNR.shape)

SNR matrix: (1839, 24)


Effective bandwidth and range resolution

For the first implementation:

$$ B_e\approx B $$

and:

$$ \boxed{ \Delta R=\frac{c}{2B_e} } $$

Later, when you model the actual windowed waveform spectrum, replace \(B_e=B\) with the measured effective bandwidth.

In [76]:
effective_bandwidth = B.copy()

range_resolution = C / (2 * effective_bandwidth)

# Energy estimate
energy = Amp**2 * Tp

ACF / PSL / ISL

For every candidate waveform:

$$ ACF(\tau)= \int x(t)x^*(t-\tau)\,dt $$

We then obtain:

$$ PSL= 20\log_{10} \left( \frac{A_{side}} {A_{main}} \right) $$

and:

$$ ISL= 10\log_{10} \left( \frac{E_{side}} {E_{main}} \right) $$

For the first LUT-generation version, these can be calculated from the candidate waveform itself because they are primarily waveform-design characteristics.

In [77]:
# ============================================================
# CELL — ACF / PSL / ISL metrics for 24 candidates
# ============================================================

# Approximate waveform correlation metrics for the 24-candidate LUT.
#
# PSL  = Peak Sidelobe Level
# ISL  = Integrated Sidelobe Level
# ACF  = Autocorrelation Function
#
# These values are still approximate.
# Later, replace them with metrics calculated from actual
# sampled LFM / GSFM / Phase-coded waveforms.

PSL_VALUES = np.array([
    # ---------------- LFM ----------------
    -13.0,   # C01
    -13.0,   # C02
    -13.0,   # C03
    -13.0,   # C04
    -13.0,   # C05
    -13.0,   # C06
    -13.0,   # C07
    -13.0,   # C08

    # ---------------- GSFM ----------------
    -18.0,   # C09
    -18.0,   # C10
    -18.0,   # C11
    -18.0,   # C12
    -18.0,   # C13
    -18.0,   # C14
    -18.0,   # C15
    -18.0,   # C16

    # ---------------- Phase-coded ----------------
    -17.0,   # C17
    -17.0,   # C18
    -17.0,   # C19
    -22.0,   # C20
    -22.0,   # C21
    -22.0,   # C22
    -22.0,   # C23
    -22.0    # C24
])


ISL_VALUES = np.array([
    # ---------------- LFM ----------------
    -8.0,    # C01
    -8.0,    # C02
    -8.0,    # C03
    -8.0,    # C04
    -8.0,    # C05
    -8.0,    # C06
    -8.0,    # C07
    -8.0,    # C08

    # ---------------- GSFM ----------------
    -12.0,   # C09
    -12.0,   # C10
    -12.0,   # C11
    -12.0,   # C12
    -12.0,   # C13
    -12.0,   # C14
    -12.0,   # C15
    -12.0,   # C16

    # ---------------- Phase-coded ----------------
    -11.0,   # C17
    -11.0,   # C18
    -11.0,   # C19
    -14.0,   # C20
    -14.0,   # C21
    -14.0,   # C22
    -14.0,   # C23
    -14.0    # C24
])


# ------------------------------------------------------------
# Create ocean-state × candidate matrices
# ------------------------------------------------------------

psl_matrix = np.tile(
    PSL_VALUES,
    (len(df), 1)
)

isl_matrix = np.tile(
    ISL_VALUES,
    (len(df), 1)
)


# ------------------------------------------------------------
# Approximate correlation coefficient
# ------------------------------------------------------------
#
# Higher SNR -> stronger/more reliable matched-filter
# correlation.
#
# CC is constrained to [0, 1].

CC = (
    1.0
    - np.exp(
        -np.maximum(SNR, 0) / 10.0
    )
)

CC = np.clip(
    CC,
    0,
    1
)


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("Number of candidates:", len(PSL_VALUES))
print("PSL matrix shape:", psl_matrix.shape)
print("ISL matrix shape:", isl_matrix.shape)
print("CC matrix shape:", CC.shape)

Number of candidates: 24
PSL matrix shape: (1839, 24)
ISL matrix shape: (1839, 24)
CC matrix shape: (1839, 24)


Candidate score

Now we combine the important quantities.

A simple optimization objective is:

$$ J_k= w_1SNR_k -w_2E_k -w_3PSL_k^{*} -w_4ISL_k^{*} -w_5\Delta R_k $$

subject to:

$$ D_k\ge D_{min} $$ $$ \Delta R_k\le\Delta R_{max} $$ $$ E_k\le E_{max} $$

and hardware frequency/bandwidth constraints.

The weights are engineering choices.

For your first prototype, we'll prioritize:

Detection margin
Range resolution
Low energy
Low sidelobes

In [78]:
# ==============================
# CANDIDATE CONSTRAINTS + OPTIMIZATION
# ==============================

# ------------------------------------------------
# 1. Check candidate dimensions
# ------------------------------------------------

NUM_CANDIDATES = 24

assert F.shape[1] == NUM_CANDIDATES, \
    f"Expected {NUM_CANDIDATES} candidates, got {F.shape[1]}"

assert B.shape[1] == NUM_CANDIDATES
assert SNR.shape[1] == NUM_CANDIDATES
assert energy.shape[1] == NUM_CANDIDATES
assert range_resolution.shape[1] == NUM_CANDIDATES


# ------------------------------------------------
# 2. Candidate validity constraints
# ------------------------------------------------

valid = (
    (detection_margin >= 0)
    &
    (F >= MIN_FREQUENCY_HZ)
    &
    (F <= MAX_FREQUENCY_HZ)
    &
    (B >= MIN_BANDWIDTH_HZ)
    &
    (B <= MAX_BANDWIDTH_HZ)
    &
    (energy <= MAX_ENERGY_J)
)


# ------------------------------------------------
# 3. Normalize SNR
# ------------------------------------------------

snr_norm = np.clip(
    SNR / 50.0,
    -2,
    2
)


# ------------------------------------------------
# 4. Normalize range resolution
#
# Smaller ΔR = better resolution
#
# ΔR = c / (2B)
# ------------------------------------------------

resolution_norm = np.clip(
    1.0 / np.maximum(range_resolution, 1e-6) / 100.0,
    0,
    2
)


# ------------------------------------------------
# 5. Normalize energy
#
# Lower energy = better
# ------------------------------------------------

energy_norm = np.clip(
    energy,
    0,
    1
)


# ------------------------------------------------
# 6. PSL / ISL contribution
#
# More negative dB = better
# ------------------------------------------------

psl_bonus = np.clip(
    -psl_matrix / 30.0,
    0,
    1
)

isl_bonus = np.clip(
    -isl_matrix / 30.0,
    0,
    1
)


# ------------------------------------------------
# 7. Overall engineering score
# ------------------------------------------------
#
# Higher score = better candidate
#
# Detection/SNR       : 50%
# Range resolution    : 20%
# PSL                 : 10%
# ISL                 : 10%
# Energy penalty      : 10%
#

score = (
    0.50 * snr_norm
    + 0.20 * resolution_norm
    + 0.10 * psl_bonus
    + 0.10 * isl_bonus
    - 0.10 * energy_norm
)


# ------------------------------------------------
# 8. Reject invalid candidates
# ------------------------------------------------

score[~valid] = -np.inf


# ------------------------------------------------
# 9. Select best candidate for each ocean state
# ------------------------------------------------

best_idx = np.argmax(
    score,
    axis=1
)


# ------------------------------------------------
# 10. Check that a valid candidate exists
# ------------------------------------------------

has_valid_candidate = np.any(
    valid,
    axis=1
)

print("Candidate selection complete.")
print("Total candidates:", NUM_CANDIDATES)
print("Ocean observations:", len(df))
print(
    "Observations with at least one valid candidate:",
    np.sum(has_valid_candidate),
    "/",
    len(df)
)

Candidate selection complete.
Total candidates: 24
Ocean observations: 1839
Observations with at least one valid candidate: 0 / 1839


## Selected Candidate

For each real ocean observation, select the candidate waveform
with the highest feasible acoustic-performance score.

\[
k^*=\arg\max_k J_k
\]

subject to all detection and hardware constraints.

In [79]:
df["candidate_id"] = candidates["candidate_id"].values[best_idx]

## Waveform Family

The selected candidate determines the waveform family:

LFM, GSFM, Phase-coded, or Safe-Fallback.

In [80]:
df["waveform_type"] = candidates["waveform_type"].values[best_idx]

## Centre Frequency

\[
f_c=f_{c,k^*}
\]

The centre frequency is selected because it produces an acceptable
trade-off between propagation loss, noise, detection margin and
hardware constraints.

In [81]:
df["center_frequency_Hz"] = candidates["center_frequency_hz"].values[best_idx]

## Bandwidth

\[
B=B_{k^*}
\]

Bandwidth affects both detection processing gain and range resolution.

In [82]:
df["bandwidth_Hz"] = candidates["bandwidth_hz"].values[best_idx]

## Pulse Duration

\[
T_p=T_{p,k^*}
\]

Pulse duration contributes to processing gain:

\[
G_p\approx10\log_{10}(BT_p)
\]

In [83]:
df["pulse_duration_s"] = candidates["pulse_duration_s"].values[best_idx]

## Normalized Transmit Amplitude

The current implementation stores normalized amplitude:

\[
0\le A\le1
\]

This should later be calibrated against the actual DAC/transducer
voltage and acoustic source level.

In [84]:
df["amplitude_normalized"] = candidates["amplitude"].values[best_idx]

## Frequency-dependent Acoustic Absorption

\[
\alpha=\alpha(f_c)
\]

The selected candidate's centre frequency determines the absorption.

In [85]:
selected_alpha = candidates["absorption_dB_per_km"].values[best_idx]

df["absorption_dB_per_km_selected"] = selected_alpha

## Transmission Loss

\[
TL=
20\log_{10}(R)
+
\alpha\frac{R}{1000}
+
\eta_s
\]

For active sonar the propagation occurs on the outgoing and returning
paths, so the active sonar equation uses approximately \(2TL\).

In [86]:
selected_tl = TOTAL_TL[
    np.arange(len(df)),
    best_idx
]

df["transmission_loss_dB_selected"] = selected_tl

## Received Signal Level

For the active-sonar model:

\[
RL=SL-2TL+TS
\]


In [87]:
selected_rl = RL[
    np.arange(len(df)),
    best_idx
]

df["received_signal_level_dB"] = selected_rl

## Processed Signal-to-Noise Ratio

\[
SNR=
RL-NL_B+10\log_{10}(BT_p)
\]

This is the estimated processed SNR for the selected candidate.

In [88]:
selected_snr = SNR[
    np.arange(len(df)),
    best_idx
]

df["snr_dB"] = selected_snr

## Detection Margin

\[
D_k=SNR_k-D_{\min}
\]

Positive values indicate that the candidate exceeds the configured
minimum detection margin.

In [89]:
df["detection_margin_dB"] = (
    selected_snr - MIN_DETECTION_MARGIN_DB
)

## Effective Bandwidth

For this first implementation:

\[
B_e\approx B
\]

Later, calculate \(B_e\) directly from the generated waveform spectrum.

In [90]:
df["effective_bandwidth_Hz"] = (
    candidates["bandwidth_hz"].values[best_idx]
)

## Range Resolution

\[
\boxed{
\Delta R=\frac{c}{2B_e}
}
\]

This directly connects your measured sound speed to the selected waveform.

In [91]:
selected_resolution = range_resolution[
    np.arange(len(df)),
    best_idx
]

df["range_resolution_m"] = selected_resolution

## Peak Sidelobe Level

\[
PSL=
20\log_{10}
\left(
\frac{A_{sidelobe}}
{A_{main}}
\right)
\]

More negative PSL generally means lower sidelobes.

In [92]:
df["PSL_dB"] = psl_matrix[
    np.arange(len(df)),
    best_idx
]

## Integrated Sidelobe Level

\[
ISL=
10\log_{10}
\left(
\frac{E_{sidelobes}}
{E_{main}}
\right)
\]

In [93]:
df["ISL_dB"] = isl_matrix[
    np.arange(len(df)),
    best_idx
]

## Correlation Coefficient

The received waveform is compared with the expected transmitted waveform.

For this initial LUT version we use the SNR-dependent correlation estimate.
The final implementation should calculate this from the actual simulated
received waveform.

In [94]:
df["correlation_coefficient"] = CC[
    np.arange(len(df)),
    best_idx
]

## Estimated Pulse Energy

For normalized amplitude:

\[
E_J \propto A^2T_p
\]

For the real hardware:

\[
\boxed{
E_J=\int V(t)I(t)\,dt
}
\]

The current column is therefore an estimate, not a hardware measurement.

In [95]:
df["estimated_energy_J"] = energy[0, best_idx]

## Hardware Validity

A candidate is valid only if:

\[
D_k\ge D_{\min}
\]

\[
B_k\in[B_{\min},B_{\max}]
\]

\[
f_c\in[f_{\min},f_{\max}]
\]

\[
E_k\le E_{\max}
\]

In [96]:
selected_valid = valid[
    np.arange(len(df)),
    best_idx
]

df["hardware_valid"] = selected_valid.astype(bool)

In [97]:
df.head()

,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m,pressure_dbar,absorption_dB_per_km,...,received_signal_level_dB,snr_dB,detection_margin_dB,effective_bandwidth_Hz,range_resolution_m,PSL_dB,ISL_dB,correlation_coefficient,estimated_energy_J,hardware_valid
0,1.0,14.41920,24.17840,0.5541,1491.957290,1473.875,18.082290,21.0,0.099010,34.068663,...,83.701168,-1.223922,-11.223922,10000,0.074598,-13.0,-8.0,0.0,0.0032,False
1,2.0,14.51465,34.82145,0.5238,1504.968400,1473.875,31.093400,21.0,0.198020,34.068663,...,83.702576,-1.137004,-11.137004,10000,0.075248,-13.0,-8.0,0.0,0.0032,False
2,5.0,14.26420,35.93365,0.7444,1505.540125,1473.875,31.665125,21.0,0.495050,34.068663,...,83.692324,-1.734437,-11.734437,10000,0.075277,-13.0,-8.0,0.0,0.0032,False
3,10.0,14.11650,35.96055,0.8524,1505.178267,1473.875,31.303267,21.0,0.990099,34.068663,...,83.687304,-2.000344,-12.000344,10000,0.075259,-13.0,-8.0,0.0,0.0032,False
4,20.0,14.07870,35.98490,1.2605,1505.248443,1473.875,31.373443,21.0,1.980198,34.068663,...,83.668337,-2.884008,-12.884008,10000,0.075262,-13.0,-8.0,0.0,0.0032,False


In [98]:
df.describe()

,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m,pressure_dbar,absorption_dB_per_km,...,transmission_loss_dB_selected,received_signal_level_dB,snr_dB,detection_margin_dB,effective_bandwidth_Hz,range_resolution_m,PSL_dB,ISL_dB,correlation_coefficient,estimated_energy_J
count,1839.000000,1839.000000,1839.000000,1839.000000,1839.000000,1839.000000,1839.000000,1839.000000,1839.000000,1.839000e+03,...,1839.000000,1839.000000,1839.000000,1839.000000,1839.0,1839.000000,1839.0,1839.0,1839.000000,1.839000e+03
mean,244.186786,18.237474,34.950096,1.003337,1518.340730,1504.629141,13.711589,1137.828199,24.176910,3.406866e+01,...,42.190755,83.680289,-1.823319,-11.823319,10000.0,0.075917,-13.0,-8.0,0.000026,3.200000e-03
std,506.620113,7.214249,3.053257,3.677794,16.783648,16.902848,19.382935,1211.166045,50.160407,6.680918e-13,...,0.085464,0.170928,1.449822,1.449822,0.0,0.000839,0.0,0.0,0.000761,6.897401e-17
min,0.500000,1.074628,4.504050,0.095900,1473.994739,1385.130000,-25.336318,9.000000,0.049505,3.406866e+01,...,42.169668,79.084406,-23.964418,-33.964418,10000.0,0.073700,-13.0,-8.0,0.000000,3.200000e-03
25%,10.000000,12.992215,34.935500,0.633350,1504.153614,1496.785000,0.000000,156.000000,0.990099,3.406866e+01,...,42.182158,83.688545,-1.936049,-11.936049,10000.0,0.075208,-13.0,-8.0,0.000000,3.200000e-03
50%,50.000000,19.498948,35.695660,0.727600,1521.457548,1504.482551,8.152869,795.000000,4.950495,3.406866e+01,...,42.184348,83.693104,-1.691628,-11.691628,10000.0,0.076073,-13.0,-8.0,0.000000,3.200000e-03
75%,200.000000,23.780825,36.267600,0.825700,1532.155003,1513.503743,28.864935,1500.000000,19.801980,3.406866e+01,...,42.186627,83.697485,-1.443608,-11.443608,10000.0,0.076608,-13.0,-8.0,0.000000,3.200000e-03
max,5000.000000,35.208053,39.003450,99.891000,1558.807995,1547.610000,117.302821,5140.600000,495.049505,3.406866e+01,...,44.488697,83.722463,0.314454,-9.685546,10000.0,0.077940,-13.0,-8.0,0.030956,3.200000e-03


In [99]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1839 entries, 0 to 1838
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   depth_m                        1839 non-null   float64
 1   temperature_C                  1839 non-null   float64
 2   salinity_PSU                   1839 non-null   float64
 3   turbidity_NTU                  1839 non-null   float64
 4   sound_speed_mps_mackenzie      1839 non-null   float64
 5   sound_speed_data_mps           1839 non-null   float64
 6   sound_speed_delta_mps          1839 non-null   float64
 7   profile_depth_max_m            1839 non-null   float64
 8   pressure_dbar                  1839 non-null   float64
 9   absorption_dB_per_km           1839 non-null   float64
 10  transmission_loss_dB           1839 non-null   float64
 11  scattering_index               1839 non-null   float64
 12  candidate_id                   1839 non-null   o

In [100]:
(df['hardware_valid'] == True).sum()

np.int64(0)

In [101]:
# Save the final dataset
output_path = "/content/real_ocean_data.csv"

df.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Final shape:", df.shape)

display(df.head())

Saved: /content/real_ocean_data.csv
Final shape: (1839, 30)


,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m,pressure_dbar,absorption_dB_per_km,...,received_signal_level_dB,snr_dB,detection_margin_dB,effective_bandwidth_Hz,range_resolution_m,PSL_dB,ISL_dB,correlation_coefficient,estimated_energy_J,hardware_valid
0,1.0,14.41920,24.17840,0.5541,1491.957290,1473.875,18.082290,21.0,0.099010,34.068663,...,83.701168,-1.223922,-11.223922,10000,0.074598,-13.0,-8.0,0.0,0.0032,False
1,2.0,14.51465,34.82145,0.5238,1504.968400,1473.875,31.093400,21.0,0.198020,34.068663,...,83.702576,-1.137004,-11.137004,10000,0.075248,-13.0,-8.0,0.0,0.0032,False
2,5.0,14.26420,35.93365,0.7444,1505.540125,1473.875,31.665125,21.0,0.495050,34.068663,...,83.692324,-1.734437,-11.734437,10000,0.075277,-13.0,-8.0,0.0,0.0032,False
3,10.0,14.11650,35.96055,0.8524,1505.178267,1473.875,31.303267,21.0,0.990099,34.068663,...,83.687304,-2.000344,-12.000344,10000,0.075259,-13.0,-8.0,0.0,0.0032,False
4,20.0,14.07870,35.98490,1.2605,1505.248443,1473.875,31.373443,21.0,1.980198,34.068663,...,83.668337,-2.884008,-12.884008,10000,0.075262,-13.0,-8.0,0.0,0.0032,False
